# Railway Aggregations
- [Railway Aggregations](#railway-aggregations)
- [Nearest Malls](#nearest-malls)
- [Traffic](#traffic)
- [House Nearest Stations](#house-nearest-stations)
- [Conclusion](#conclusion)

Written By: Keltonn [A0222591J]

AI Used: Claude Sonnet 5.5 for API Lookup, Data Scraping

In this notebook, we will be doing feature engineering for the railway features.

### Feature Types

There are two general types of features for this problem:

|Feature Type|Description|
|---|---|
| Temporal Features | Features that change every month, like stock market data and Economic Indicators |
| Static Features | Features that have no trend, don't change. Includes most location features, such as housing type |

### Task

For this notebook, we will be processing the features related to the nearby railway stations for each HDB Block.

In [29]:
import pandas as pd
import numpy as np
import scipy as scp
import math
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
import os, sys
from pathlib import Path
_here = Path(os.environ.get('RENT_PROJECT_ROOT') or Path.cwd()).resolve()
sys.path.insert(0, str(next(p for p in [_here, *_here.parents] if (p / 'data' / 'config.py').exists())))      # final_report: data/config.py holds every path
from data import config as cfg
ROOT = cfg.ROOT

# Nearest Malls
- [Railway Aggregations](#railway-aggregations)
- [Nearest Malls](#nearest-malls)
- [Traffic](#traffic)
- [House Nearest Stations](#house-nearest-stations)
- [Conclusion](#conclusion)


Malls often are built around railway stations in Singapore (Or is it the other way round? Who knows!), so it is good to get the nearest malls for each station. Claude has already processed this for us during the data collection and data scraping stage, so all we need to do is join it to the existing MRT nodes.

In [7]:
mrt_node = pd.read_csv(f'{cfg.RAW_GRAPH_DIR}/sg-mrt-nodes.csv')
mall_feature = pd.read_csv(f'{cfg.MRT_DATA_DIR}/sg-station-mall-counts.csv')
mall_feature.columns = ['NAME','n_malls_500m','nearest_mall','nearest_mall_m','malls_500m']
mall_feature = mall_feature[['NAME','n_malls_500m','malls_500m','nearest_mall','nearest_mall_m',]]
df = pd.merge(mrt_node, mall_feature, how = 'left', on = 'NAME')
df

,NAME,member_codes,lines,n_lines,status,latitude,longitude,subzone,planning_area,region,is_interchange,n_malls_500m,malls_500m,nearest_mall,nearest_mall_m
0,admiralty mrt station,ns10,ns,1,open,1.440689,103.800933,woodlands east,woodlands,north region,False,0,NaN,Causeway Point,1739.6
1,aljunied mrt station,ew9,ew,1,open,1.316389,103.882778,aljunied,geylang,central region,False,0,NaN,Lot 1,979.0
2,ang mo kio mrt station,cr11;ns16,cr;ns,2,open,1.370017,103.849450,cheng san,ang mo kio,north-east region,True,1,AMK Hub,AMK Hub,128.6
3,aviation park mrt station,cr2,cr,1,open,1.370000,104.003056,changi airport,changi,east region,False,0,NaN,Jewel Changi Airport,1889.0
4,bahar junction mrt station,js7,js,1,open,1.345422,103.702542,jurong west central,jurong west,west region,False,0,NaN,Gek Poh Shopping Centre,649.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,woodleigh mrt station,ne11,ne,1,open,1.339167,103.870833,bidadari,toa payoh,central region,False,0,NaN,NEX,1300.3
188,xilin mrt station,dt36,dt,1,open,1.328889,103.965000,xilin,tampines,east region,False,0,NaN,Changi City Point,617.4
189,yew tee mrt station,ns5,ns,1,open,1.397500,103.747500,yew tee,choa chu kang,west region,False,1,Yew Tee Point,Yew Tee Point,68.8
190,yio chu kang mrt station,ns15,ns,1,open,1.381906,103.844817,yio chu kang,ang mo kio,north-east region,False,0,NaN,AMK Hub,1450.1


In [9]:
df.to_csv(f'{cfg.LOCATION_MINING_DIR}/station-keys.csv', index = False)

# Traffic
- [Railway Aggregations](#railway-aggregations)
- [Nearest Malls](#nearest-malls)
- [Traffic](#traffic)
- [House Nearest Stations](#house-nearest-stations)
- [Conclusion](#conclusion)


We have collected some critical data on the foot traffic for each railway station. This is useful in determining the popularity of a station. The property value of a house is potentially tied to the proximity to work locations, tourist hotspots etc. Foot traffic of railway stations is a good proxy to that information.

Claude has already collected this data for us in the public dataset and dropped any railway stations that have missing values for any of the features.

In [11]:
dates = ['202605','202606','202607']
traffic_lst = []
for d in dates:
    traffic = pd.read_csv(f'{cfg.MRT_DATA_DIR}/sg-mrt-nodes-features-{d}.csv')
    traffic = traffic[traffic['pv_matched'] == True].reset_index(drop = True)
    traffic = traffic[['NAME',
        'pv_tap_in', 'pv_tap_out', 'pv_total', 'pv_weekday_total',
        'pv_weekend_total', 'pv_am_peak_total', 'pv_pm_peak_total',
        'pv_busiest_hour', 'year_month',]]
    traffic_lst.append(traffic)

traffic_df = pd.concat(traffic_lst).reset_index(drop = True)
traffic_df.to_csv(f'{cfg.LOCATION_MINING_DIR}/station-traffic.csv', index= False)
traffic_df


,NAME,pv_tap_in,pv_tap_out,pv_total,pv_weekday_total,pv_weekend_total,pv_am_peak_total,pv_pm_peak_total,pv_busiest_hour,year_month
0,admiralty mrt station,1220591.0,1088699.0,2309290.0,1601448.0,707842.0,447168.0,423421.0,7.0,2026-05
1,aljunied mrt station,701918.0,708113.0,1410031.0,874144.0,535887.0,210393.0,239064.0,18.0,2026-05
2,ang mo kio mrt station,1220599.0,1339014.0,2559613.0,1726662.0,832951.0,408605.0,466877.0,18.0,2026-05
3,bartley mrt station,254624.0,228521.0,483145.0,353537.0,129608.0,101289.0,86506.0,7.0,2026-05
4,bayfront mrt station,1024518.0,966289.0,1990807.0,1046697.0,944110.0,133522.0,254453.0,20.0,2026-05
...,...,...,...,...,...,...,...,...,...,...
427,woodlands south mrt station,402335.0,373959.0,776294.0,637179.0,139115.0,176196.0,164702.0,7.0,2026-07
428,woodleigh mrt station,467808.0,480973.0,948781.0,741200.0,207581.0,167851.0,211816.0,18.0,2026-07
429,yew tee mrt station,919637.0,886547.0,1806184.0,1434612.0,371572.0,376052.0,383765.0,7.0,2026-07
430,yio chu kang mrt station,687769.0,640383.0,1328152.0,1157293.0,170859.0,346223.0,308066.0,8.0,2026-07


Now we can do some basic aggregation for the Traffic. What we have above is the tap-in/tap-out value for each railway station in the months of 2026-05, 2026-06 and 2026-07. This seems like temporal data that changes over time, however the railway company does not keep a record of traffic data long enough to fit the dataset, so we can only treat this as a snapshot popularity of each railway station.

Therefore, for each station, we take the latest month, 2026-07-01, and perform standard scaling on all the traffic features. This will return some kind of ranking of stations, where very busy stations will have positive value, and very sparse stations will have negative value.

There is one categorical feature, `pv_busiest_hour`, which is the time of day where a station is most busy. We will perform one-hot encoding later before feeding it into the model training stage.

In [18]:
# Define feature columns by numerical/categorical
num_col = ['pv_tap_in', 'pv_tap_out', 'pv_total', 'pv_weekday_total',
       'pv_weekend_total', 'pv_am_peak_total', 'pv_pm_peak_total',
       ]
cat_col = ['pv_busiest_hour',]

# Get latest traffic data
traffic_df = pd.read_csv(f'{cfg.LOCATION_MINING_DIR}/station-traffic.csv')
traffic_df = traffic_df[traffic_df['year_month'] == '2026-07']
tdf = traffic_df.copy(deep = True)
# Scale dataset
try:
    scale = joblib.load(f'{cfg.TRAFFIC_SCALER_PKL}')
    tdf[num_col] = scale.transform(traffic_df[num_col])
except:
       print('No prior Railway station traffic scaler detected, training new one')
       scale = StandardScaler()
       tdf[num_col] = scale.fit_transform(traffic_df[num_col])
       joblib.dump(scale, f'{cfg.TRAFFIC_SCALER_PKL}')
tdf = tdf.drop('year_month', axis = 1).reset_index(drop = True)
tdf.columns = ['NAME', 'mrt_traffic_tap_in_amount_norm', 'mrt_traffic_tap_out_amount_norm', 'mrt_traffic_total_tap_norm', 'mrt_traffic_weekday_tap_norm',
       'mrt_traffic_weekend_tap_norm', 'mrt_traffic_am_peak_norm', 'mrt_traffic_pm_peak_norm',
       'mrt_traffic_busiest_hour']
tdf.to_csv(f'{cfg.LOCATION_MINING_DIR}/station-traffic-std.csv', index = False)
tdf

,NAME,mrt_traffic_tap_in_amount_norm,mrt_traffic_tap_out_amount_norm,mrt_traffic_total_tap_norm,mrt_traffic_weekday_tap_norm,mrt_traffic_weekend_tap_norm,mrt_traffic_am_peak_norm,mrt_traffic_pm_peak_norm,mrt_traffic_busiest_hour
0,admiralty mrt station,1.397545,1.058514,1.227654,1.291102,0.943364,1.782587,1.098732,7.0
1,aljunied mrt station,0.229524,0.239906,0.235274,0.154663,0.472060,0.137620,0.115453,18.0
2,ang mo kio mrt station,1.350448,1.572875,1.466466,1.494068,1.278170,1.521358,1.346194,18.0
3,bartley mrt station,-0.663381,-0.705300,-0.686068,-0.672864,-0.679976,-0.607448,-0.717120,7.0
4,bayfront mrt station,1.076316,0.875100,0.975982,0.665476,1.883226,-0.366818,0.376355,20.0
...,...,...,...,...,...,...,...,...,...
139,woodlands south mrt station,-0.430875,-0.471241,-0.452304,-0.420092,-0.522092,-0.314457,-0.473874,7.0
140,woodleigh mrt station,-0.297136,-0.259848,-0.278739,-0.284393,-0.241669,-0.361854,-0.262955,18.0
141,yew tee mrt station,0.625798,0.541313,0.584020,0.620183,0.430005,0.820654,0.506821,7.0
142,yio chu kang mrt station,0.152170,0.055046,0.103002,0.258412,-0.392075,0.651236,0.167934,8.0


# House Nearest Stations
- [Railway Aggregations](#railway-aggregations)
- [Nearest Malls](#nearest-malls)
- [Traffic](#traffic)
- [House Nearest Stations](#house-nearest-stations)
- [Conclusion](#conclusion)


Here we get the number of railway stations within 500m, 1000m, the name of the nearest train station and the distance to that nearest train station. We will be using all of this in our model training later.

In [19]:
LAT0 = 1.35
R_EARTH = 6371000.0
def to_xy(lat, lon):
    x = np.radians(np.asarray(lon, float)) * math.cos(math.radians(LAT0)) * R_EARTH
    y = np.radians(np.asarray(lat, float)) * R_EARTH
    return np.column_stack([np.ravel(x), np.ravel(y)])

In [23]:
train_house = pd.read_csv(f'{cfg.DF_KEYS_DIR}/train-house.csv')
test_house = pd.read_csv(f'{cfg.DF_KEYS_DIR}/test-house.csv')
houses = pd.concat(
        [train_house, test_house]
    )[
        ['TOWN','BLOCK','STREET','POSTAL_CODE','LATITUDE','LONGITUDE']
    ].drop_duplicates().reset_index(drop = True)
stations = pd.read_csv(f'{cfg.LOCATION_MINING_DIR}/station-keys.csv')[['NAME','latitude','longitude']]

In [26]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(stations.latitude.values, stations.longitude.values)

# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

link_rows, count_rows = [], []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_station": stations.NAME[nearest],
        "nearest_station_m": round(float(d[nearest]), 1),
    }

    for WALK_M in [500,1000]:
        within = np.where(d <= WALK_M)[0]
        order = within[np.argsort(d[within])]  # nearest first
        for rank, m in enumerate(order, start=1):
            link_rows.append({
                "house_town": houses.TOWN[s], "house_block": houses.BLOCK[s], "house_street": houses.STREET[s], "station_name": stations.NAME[m],
                "distance_m": round(float(d[m]), 1), "rank": rank,
            })
        results[f"n_stations_{int(WALK_M)}m"] = len(order)
        # results[f"stations_{int(WALK_M)}m"]=  ";".join(stations.NAME[order].tolist())

    count_rows.append(results)

links = pd.DataFrame(link_rows).sort_values(["station_name", "rank"])
counts = pd.DataFrame(count_rows)
counts.to_csv(f'{cfg.LOCATION_MINING_DIR}/hdb_nearest_station.csv', index = False)
display(counts)

,TOWN,BLOCK,STREET,nearest_station,nearest_station_m,n_stations_500m,n_stations_1000m
0,punggol,604a,punggol road,punggol mrt/lrt station,556.7,0,1
1,sengkang,290c,compassvale crescent,sengkang mrt/lrt station,634.4,0,1
2,queenstown,5,ghim moh road,buona vista mrt station,625.9,0,2
3,sengkang,436b,fernvale road,sengkang mrt/lrt station,2069.8,0,0
4,sembawang,467b,admiralty drive,sembawang mrt station,762.3,0,1
...,...,...,...,...,...,...,...
9451,kallang/whampoa,111,lorong 3 geylang,kallang mrt station,278.7,1,1
9452,yishun,753,yishun street 72,yishun mrt station,283.7,1,1
9453,yishun,676c,yishun ring road,khatib mrt station,1167.8,0,0
9454,tampines,606d,tampines street 61,tampines north mrt station,817.3,0,1


# Conclusion
- [Railway Aggregations](#railway-aggregations)
- [Nearest Malls](#nearest-malls)
- [Traffic](#traffic)
- [House Nearest Stations](#house-nearest-stations)
- [Conclusion](#conclusion)

We produced three sets of tables relating to railway stations:
1. Nearest mall per railway station
2. Foot traffic per railway station
3. Number of railway stations per house

In [27]:
malls = pd.read_csv(f'{cfg.LOCATION_MINING_DIR}/station-keys.csv')
traffic = pd.read_csv(f'{cfg.LOCATION_MINING_DIR}/station-traffic-std.csv')
near_stat = pd.read_csv(f'{cfg.LOCATION_MINING_DIR}/hdb_nearest_station.csv')
display(malls)
display(traffic)
display(near_stat)

,NAME,member_codes,lines,n_lines,status,latitude,longitude,subzone,planning_area,region,is_interchange,n_malls_500m,malls_500m,nearest_mall,nearest_mall_m
0,admiralty mrt station,ns10,ns,1,open,1.440689,103.800933,woodlands east,woodlands,north region,False,0,NaN,Causeway Point,1739.6
1,aljunied mrt station,ew9,ew,1,open,1.316389,103.882778,aljunied,geylang,central region,False,0,NaN,Lot 1,979.0
2,ang mo kio mrt station,cr11;ns16,cr;ns,2,open,1.370017,103.849450,cheng san,ang mo kio,north-east region,True,1,AMK Hub,AMK Hub,128.6
3,aviation park mrt station,cr2,cr,1,open,1.370000,104.003056,changi airport,changi,east region,False,0,NaN,Jewel Changi Airport,1889.0
4,bahar junction mrt station,js7,js,1,open,1.345422,103.702542,jurong west central,jurong west,west region,False,0,NaN,Gek Poh Shopping Centre,649.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,woodleigh mrt station,ne11,ne,1,open,1.339167,103.870833,bidadari,toa payoh,central region,False,0,NaN,NEX,1300.3
188,xilin mrt station,dt36,dt,1,open,1.328889,103.965000,xilin,tampines,east region,False,0,NaN,Changi City Point,617.4
189,yew tee mrt station,ns5,ns,1,open,1.397500,103.747500,yew tee,choa chu kang,west region,False,1,Yew Tee Point,Yew Tee Point,68.8
190,yio chu kang mrt station,ns15,ns,1,open,1.381906,103.844817,yio chu kang,ang mo kio,north-east region,False,0,NaN,AMK Hub,1450.1


,NAME,mrt_traffic_tap_in_amount_norm,mrt_traffic_tap_out_amount_norm,mrt_traffic_total_tap_norm,mrt_traffic_weekday_tap_norm,mrt_traffic_weekend_tap_norm,mrt_traffic_am_peak_norm,mrt_traffic_pm_peak_norm,mrt_traffic_busiest_hour
0,admiralty mrt station,1.397545,1.058514,1.227654,1.291102,0.943364,1.782587,1.098732,7.0
1,aljunied mrt station,0.229524,0.239906,0.235274,0.154663,0.472060,0.137620,0.115453,18.0
2,ang mo kio mrt station,1.350448,1.572875,1.466466,1.494068,1.278170,1.521358,1.346194,18.0
3,bartley mrt station,-0.663381,-0.705300,-0.686068,-0.672864,-0.679976,-0.607448,-0.717120,7.0
4,bayfront mrt station,1.076316,0.875100,0.975982,0.665476,1.883226,-0.366818,0.376355,20.0
...,...,...,...,...,...,...,...,...,...
139,woodlands south mrt station,-0.430875,-0.471241,-0.452304,-0.420092,-0.522092,-0.314457,-0.473874,7.0
140,woodleigh mrt station,-0.297136,-0.259848,-0.278739,-0.284393,-0.241669,-0.361854,-0.262955,18.0
141,yew tee mrt station,0.625798,0.541313,0.584020,0.620183,0.430005,0.820654,0.506821,7.0
142,yio chu kang mrt station,0.152170,0.055046,0.103002,0.258412,-0.392075,0.651236,0.167934,8.0


,TOWN,BLOCK,STREET,nearest_station,nearest_station_m,n_stations_500m,n_stations_1000m
0,punggol,604a,punggol road,punggol mrt/lrt station,556.7,0,1
1,sengkang,290c,compassvale crescent,sengkang mrt/lrt station,634.4,0,1
2,queenstown,5,ghim moh road,buona vista mrt station,625.9,0,2
3,sengkang,436b,fernvale road,sengkang mrt/lrt station,2069.8,0,0
4,sembawang,467b,admiralty drive,sembawang mrt station,762.3,0,1
...,...,...,...,...,...,...,...
9451,kallang/whampoa,111,lorong 3 geylang,kallang mrt station,278.7,1,1
9452,yishun,753,yishun street 72,yishun mrt station,283.7,1,1
9453,yishun,676c,yishun ring road,khatib mrt station,1167.8,0,0
9454,tampines,606d,tampines street 61,tampines north mrt station,817.3,0,1
